# TechMind - Organización Inteligente del Conocimiento Técnico
## Auditoría, limpieza y enriquecimiento inicial del dataset

Este notebook realiza la auditoría, limpieza y preparación inicial del dataset del proyecto TechMind. Su objetivo es identificar el estado real de los datos, detectar ruido, valores nulos, duplicados y particularidades del texto, para generar una versión limpia, consistente y estructurada que sirva como base para los siguientes módulos del proyecto. Además, los datos quedan preparados para su posterior uso en la generación de embeddings y el motor de búsqueda semántica, facilitando la recuperación, organización y reutilización del conocimiento técnico.

##Importaciones y carga

In [51]:
import pandas as pd
import numpy as np
import re
import html
import hashlib
!pip install -q langdetect
from langdetect import detect, LangDetectException

In [52]:
df = pd.read_csv("dataset_techmind_original.csv")
df.columns = [c.strip().lower() for c in df.columns]

# Normalizar nombres de columnas
if "sourcetype" in df.columns:
    df.rename(columns={"sourcetype": "source_type"}, inplace=True)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
display(df.head())

Filas: 1008
Columnas: 3


,source_type,is_synthetic_noise,raw_text
0,Tutorial_Codigo,False,Contexto: Search for a text pattern in multipl...
1,Tutorial_Codigo,False,Contexto: Profile code execution time: {code}\...
2,Tutorial_Codigo,False,Contexto: Send a weekly scheduled email with t...
3,Articulo_Academico,False,"In this paper, we consider the nonasymptotic..."
4,Tutorial_Codigo,False,Contexto: List all running Docker containers\n...


##Auditoria

In [53]:
print("=== ESTRUCTURA GENERAL ===")
print("Dimensión:", df.shape)
print("Columnas:", list(df.columns))
print()

print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print()

print("=== VALORES NULOS ===")
print(df.isnull().sum().sort_values(ascending=False))
print()

print("=== DUPLICADOS ===")
print("Filas duplicadas:", df.duplicated().sum())
print()

print("=== MUESTRA DE REGISTROS ===")
display(df.sample(5, random_state=42))

=== ESTRUCTURA GENERAL ===
Dimensión: (1008, 3)
Columnas: ['source_type', 'is_synthetic_noise', 'raw_text']

=== TIPOS DE DATOS ===
source_type           object
is_synthetic_noise      bool
raw_text              object
dtype: object

=== VALORES NULOS ===
source_type           0
is_synthetic_noise    0
raw_text              0
dtype: int64

=== DUPLICADOS ===
Filas duplicadas: 0

=== MUESTRA DE REGISTROS ===


,source_type,is_synthetic_noise,raw_text
938,Articulo_Academico,False,We study the problem of partitioning a small...
630,Articulo_Academico,False,"In this paper, we propose a special fusion m..."
682,Articulo_Academico,False,We show that Boolean functions expressible a...
514,Tutorial_Codigo,False,Contexto: Open Google Maps if I say 'I'm lost'...
365,Tutorial_Codigo,False,Contexto: Monitor the health of all hard drive...


In [54]:
# Detectar columna principal de texto
text_candidates = [c for c in df.columns if df[c].dtype == "object"]
print("Columnas de tipo texto:", text_candidates)

if "raw_text" not in df.columns:
    raise ValueError("No se encontró la columna 'raw_text'.")

text_col = "raw_text"
print("Columna de texto usada:", text_col)

Columnas de tipo texto: ['source_type', 'raw_text']
Columna de texto usada: raw_text


In [55]:
if text_col:
    df[text_col] = df[text_col].fillna("").astype(str)
    df["text_length_chars"] = df[text_col].apply(len)
    df["text_length_words"] = df[text_col].apply(lambda x: len(x.split()))

    print("=== LONGITUD DEL TEXTO ===")
    print("Promedio caracteres:", round(df["text_length_chars"].mean(), 2))
    print("Mediana caracteres:", round(df["text_length_chars"].median(), 2))
    print("Promedio palabras:", round(df["text_length_words"].mean(), 2))
    print("Mediana palabras:", round(df["text_length_words"].median(), 2))

=== LONGITUD DEL TEXTO ===
Promedio caracteres: 683.87
Mediana caracteres: 477.0
Promedio palabras: 93.98
Mediana palabras: 59.0


In [56]:
# Ruido básico
if text_col:
    urls_count = df[text_col].str.contains(r"http[s]?://|www\.", regex=True, na=False).sum()
    html_count = df[text_col].str.contains(r"<[^>]+>", regex=True, na=False).sum()

    special_chars = df[text_col].apply(lambda x: len(re.findall(r"[^\w\sáéíóúÁÉÍÓÚñÑüÜ.,;:()\-\/]", x)))

    print("=== RUIDO BÁSICO ===")
    print("Filas con URLs:", urls_count)
    print("Filas con HTML:", html_count)
    print("Promedio de caracteres especiales raros:", round(special_chars.mean(), 2))

=== RUIDO BÁSICO ===
Filas con URLs: 107
Filas con HTML: 11
Promedio de caracteres especiales raros: 18.27


##Limpieza

In [57]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [58]:
df_clean = df.copy()

# Eliminar filas vacías y duplicados
df_clean = df_clean.dropna(how="all")
df_clean = df_clean.drop_duplicates()

# Limpiar texto
if text_col:
    df_clean[f"{text_col}_clean"] = df_clean[text_col].apply(clean_text)
    df_clean[f"{text_col}_clean"] = df_clean[f"{text_col}_clean"].str.replace(r"\s+", " ", regex=True).str.strip()

    # Eliminar textos vacíos
    df_clean = df_clean[df_clean[f"{text_col}_clean"].str.len() > 0].copy()

print("Filas después de limpieza:", df_clean.shape[0])
display(df_clean.head())

Filas después de limpieza: 1008


,source_type,is_synthetic_noise,raw_text,text_length_chars,text_length_words,raw_text_clean
0,Tutorial_Codigo,False,Contexto: Search for a text pattern in multipl...,268,42,Contexto: Search for a text pattern in multipl...
1,Tutorial_Codigo,False,Contexto: Profile code execution time: {code}\...,234,27,Contexto: Profile code execution time: {code} ...
2,Tutorial_Codigo,False,Contexto: Send a weekly scheduled email with t...,524,49,Contexto: Send a weekly scheduled email with t...
3,Articulo_Academico,False,"In this paper, we consider the nonasymptotic...",925,147,"In this paper, we consider the nonasymptotic s..."
4,Tutorial_Codigo,False,Contexto: List all running Docker containers\n...,125,17,Contexto: List all running Docker containers C...


##Enriquecimiento

In [59]:
def generate_doc_id(text):
    """
    Genera un ID determinístico a partir del contenido del documento.
    El mismo texto siempre tendrá el mismo ID.
    """
    text = re.sub(r"\s+", " ", str(text).strip())
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def detect_language_safe(text):
    try:
        if not text or len(text.strip()) < 10:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"


if text_col:

    # ID estable
    df_clean["doc_id"] = (
        df_clean[f"{text_col}_clean"]
        .apply(generate_doc_id)
    )

    # Idioma
    df_clean["language"] = (
        df_clean[f"{text_col}_clean"]
        .apply(detect_language_safe)
    )

    # Longitudes
    df_clean["clean_length_chars"] = (
        df_clean[f"{text_col}_clean"]
        .str.len()
    )

    df_clean["clean_length_words"] = (
        df_clean[f"{text_col}_clean"]
        .str.split()
        .apply(len)
    )

display(
    df_clean[
        [
            "doc_id",
            "language",
            "clean_length_chars",
            "clean_length_words"
        ]
    ].head()
)

print("=== DISTRIBUCIÓN DE IDIOMA ===")
print(df_clean["language"].value_counts(dropna=False))

,doc_id,language,clean_length_chars,clean_length_words
0,dd7091be0f4a5017,en,267,42
1,7ddb13ff9b435d29,en,233,27
2,7309876e625b5443,en,491,49
3,8510cbbea81e0de3,en,922,147
4,19a1690ee2b82fc5,en,124,17


=== DISTRIBUCIÓN DE IDIOMA ===
language
en    969
fr     28
es      8
ca      2
ro      1
Name: count, dtype: int64


## Preparación para Embeddings


In [60]:
# Preparación para Embeddings

print("Preparando dataset para generación de embeddings...")

df_embeddings = df_clean.copy()

# Crear título usando las primeras palabras
df_embeddings["titulo"] = (
    df_embeddings[f"{text_col}_clean"]
    .fillna("")
    .astype(str)
    .str.split()
    .str[:6]
    .str.join(" ")
)

# Construir dataset final
dataset_embeddings = (
    df_embeddings[
        [
            "doc_id",
            "titulo",
            "source_type",
            f"{text_col}_clean",
            "language",
            "clean_length_chars",
            "clean_length_words"
        ]
    ]
    .rename(columns={f"{text_col}_clean": "texto"})
)

# Validar que no existan IDs repetidos
assert dataset_embeddings["doc_id"].is_unique, "Existen doc_id duplicados"

print("✓ IDs únicos verificados")

print("Dataset listo para la siguiente etapa (Generación de Embeddings).")
display(dataset_embeddings.head())

Preparando dataset para generación de embeddings...
✓ IDs únicos verificados
Dataset listo para la siguiente etapa (Generación de Embeddings).


,doc_id,titulo,source_type,texto,language,clean_length_chars,clean_length_words
0,dd7091be0f4a5017,Contexto: Search for a text pattern,Tutorial_Codigo,Contexto: Search for a text pattern in multipl...,en,267,42
1,7ddb13ff9b435d29,Contexto: Profile code execution time: {code},Tutorial_Codigo,Contexto: Profile code execution time: {code} ...,en,233,27
2,7309876e625b5443,Contexto: Send a weekly scheduled email,Tutorial_Codigo,Contexto: Send a weekly scheduled email with t...,en,491,49
3,8510cbbea81e0de3,"In this paper, we consider the",Articulo_Academico,"In this paper, we consider the nonasymptotic s...",en,922,147
4,19a1690ee2b82fc5,Contexto: List all running Docker containers,Tutorial_Codigo,Contexto: List all running Docker containers C...,en,124,17


##Persistencia

In [61]:
# Salida Fase 2
# Dataset limpio para auditoría y respaldo
df_clean.to_csv(
    "dataset_techmind_clean.csv",
    index=False
)

# Entrada oficial para la Fase 3 (Embeddings)
dataset_embeddings.to_csv(
    "dataset_techmind_ready.csv",
    index=False
)

print("✔ dataset_techmind_clean.csv generado")
print("✔ dataset_techmind_ready.csv generado")

✔ dataset_techmind_clean.csv generado
✔ dataset_techmind_ready.csv generado


##Control Final

In [62]:
# Control Final

print("\n========== VALIDACIÓN FINAL ==========")

print(f"\nTotal de documentos: {len(dataset_embeddings)}")
print(f"Duplicados: {dataset_embeddings.duplicated().sum()}")

print("\nValores nulos por columna:")
print(dataset_embeddings.isnull().sum())

print("\nIdiomas detectados:")
print(dataset_embeddings["language"].value_counts())

print("\nTipos de documento:")
print(dataset_embeddings["source_type"].value_counts())

print("\nEstadísticas de longitud del texto (palabras):")
print(dataset_embeddings["clean_length_words"].describe().round(2))

print("\nDataset listo para la generación de embeddings.")


========== VALIDACIÓN FINAL ==========

Total de documentos: 1008
Duplicados: 0

Valores nulos por columna:
doc_id                0
titulo                0
source_type           0
texto                 0
language              0
clean_length_chars    0
clean_length_words    0
dtype: int64

Idiomas detectados:
language
en    969
fr     28
es      8
ca      2
ro      1
Name: count, dtype: int64

Tipos de documento:
source_type
Tutorial_Codigo       500
Articulo_Academico    500
GitHub_README           5
Ruido_Web               3
Name: count, dtype: int64

Estadísticas de longitud del texto (palabras):
count    1008.00
mean       93.17
std       106.25
min         5.00
25%        32.00
50%        58.50
75%       143.00
max      2104.00
Name: clean_length_words, dtype: float64

Dataset listo para la generación de embeddings.
